# 06 · Classical ML baselines

**Adapted for current project:** Uses `features_v1.npz` from notebook 05 and the frozen `split_v2.csv`. Imputation and scaling are fit inside each CV fold. CV uses `GroupKFold` by signer; validation/test are held out. These are baseline estimates; inspect per-class support because signer-grouped folds can be sparse.


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
ROOT=Path.cwd().resolve()
while not (ROOT/'data/processed/WLASL100/features_v1.npz').exists() and ROOT.parent!=ROOT: ROOT=ROOT.parent
DATA=ROOT/'data/processed/WLASL100'
manifest=pd.read_csv(DATA/'feature_manifest_v1.csv',dtype={'video_id':str,'signer_id':str})
bundle=np.load(DATA/'features_v1.npz'); ids=bundle['video_id'].astype(str); X=bundle['X']
manifest=manifest.set_index('video_id').loc[ids].reset_index()
assert np.array_equal(manifest.video_id.to_numpy(),ids)
train=manifest.split.eq('train').to_numpy(); val=manifest.split.eq('val').to_numpy(); test=manifest.split.eq('test').to_numpy()
y=manifest.gloss.to_numpy(); groups=manifest.signer_id.to_numpy()
models={
 'logistic':make_pipeline(SimpleImputer(strategy='median',add_indicator=True),StandardScaler(),LogisticRegression(max_iter=2000,class_weight='balanced')),
 'knn':make_pipeline(SimpleImputer(strategy='median',add_indicator=True),StandardScaler(),KNeighborsClassifier(n_neighbors=5,weights='distance')),
 'random_forest':make_pipeline(SimpleImputer(strategy='median',add_indicator=True),RandomForestClassifier(n_estimators=400,class_weight='balanced_subsample',random_state=42,n_jobs=-1)),
 'rbf_svm':make_pipeline(SimpleImputer(strategy='median',add_indicator=True),StandardScaler(),SVC(C=1,class_weight='balanced')),
}
cv=GroupKFold(n_splits=3); results=[]
for name,model in models.items():
 fold=[]
 for tr,va in cv.split(X[train],y[train],groups[train]):
  model.fit(X[train][tr],y[train][tr]); pred=model.predict(X[train][va])
  fold.append((accuracy_score(y[train][va],pred),f1_score(y[train][va],pred,average='macro',zero_division=0)))
 results.append({'model':name,'cv_accuracy_mean':np.mean([a for a,b in fold]),'cv_macro_f1_mean':np.mean([b for a,b in fold])})
print(pd.DataFrame(results).sort_values('cv_macro_f1_mean',ascending=False))
